<div align="center">

#### [S. Mussard](https://sites.google.com/view/cv-stphane-mussard/accueil "Homepage") 

</div>

<div align="center">

#### Chapter 5: Multiple Gini Regressions 

</div>


<div align="center"> <a href="https://www.python.org/"><img src="https://upload.wikimedia.org/wikipedia/commons/thumb/f/f8/Python_logo_and_wordmark.svg/390px-Python_logo_and_wordmark.svg.png" style="max-width: 150px; display: inline" alt="Python"/></a> 

</div>


<div align="center"> </div>

<div align="center">

Cite: S. Mussard (2025), *Machine Learning with Gini Indices: Applications with Python*, Springer.  

</div>



# Application Gini Regression with MA(1)

In [ ]:
import numpy as np
import pandas as pd
from datetime import datetime as dt
from statsmodels.tsa.stattools import adfuller, acf, pacf
from statsmodels.tsa.arima_model import ARIMA
from statsmodels.graphics.tsaplots import plot_acf
from statsmodels.stats.stattools import durbin_watson
import statsmodels.api as sm
import matplotlib.pyplot as plt
import scipy.stats as ss
from statsmodels.tsa.arima_process import ArmaProcess
import statsmodels.tsa.api as tsa


In [ ]:
# Parameters of the model
n = 1000  
n_vars = 5    # Number of features in X
theta = 0.9   # MA(1) coefficient
outlier = 1000  # intensity of the outlier

# Generate X with outlier
X = np.random.randn(n, n_vars)
X[10] = X[10]+outlier 
X[100] = X[100]-outlier 
X[500] = X[500]+outlier 

# Generate the MA(1)
nu_t = np.random.randn(n)
ma_process = ArmaProcess([1], [1, theta])
epsilon_t = ma_process.generate_sample(nsample=n, scale=1) + nu_t  
epsilon_t[100] = epsilon_t[100]+outlier

# Generate random beta coeff and the target Y
coefficients = np.random.randn(n_vars)*2 + 10
Y = 5 + X.dot(coefficients) + epsilon_t

In [ ]:
from sklearn.model_selection import KFold
from GiniRegression_v2 import GiniRegression, Autocorrelation
from sklearn.metrics import mean_squared_error as mse

# K-fold
kf = KFold(n_splits=5, shuffle=True, random_state=1)
true_coefficients = np.append(5, coefficients)

# Initialize lists
ols_mse_list = []
gls_mse_list = []
gini_mse_list = []
estimated_ols_coeffs = []
estimated_gini_coeffs = []
estimated_gls_coeffs = []

# Train/Test
for train_index, test_index in kf.split(X):
    X_train, X_test = X[train_index], X[test_index]
    Y_train, Y_test = Y[train_index], Y[test_index]
    
    # OLS
    X_train_with_const = sm.add_constant(X_train)
    X_test_with_const = sm.add_constant(X_test)
    ols_model = sm.OLS(Y_train, X_train_with_const).fit()
    estimated_ols_coeffs.append(ols_model.params)
    ols_pred = ols_model.predict(sm.add_constant(X_test))
    ols_mse_list.append(mse(Y_test, ols_pred))
    residuals = ols_model.resid
    error_covariance = np.diag(residuals ** 2)

    # Gini Regression 
    gini_model = GiniRegression()
    gini_model.fit(Y_train, X_train)
    residuals = gini_model.residuals
    autocorr_model = Autocorrelation(GiniRegression)
    autocorr_model.fit(Y_train, X_train, residuals, autocorrel='Prais-Winsten', correlogram=True)
    estimated_gini_coeffs.append(autocorr_model.beta_coeff)
    gini_pred = autocorr_model.predict(X_test)
    gini_mse_list.append(mse(Y_test, gini_pred))
    
    # FGLS
    try:
        gls_model = sm.GLS(Y_train, X_train_with_const, sigma=error_covariance).fit()
        estimated_gls_coeffs.append(gls_model.params)
        gls_pred = gls_model.predict(sm.add_constant(X_test))  
        gls_mse_list.append(mse(Y_test, gls_pred))
    except np.linalg.LinAlgError:
        print("Skipped FGLS fitting due to a non-positive definite covariance matrix.")
        estimated_gls_coeffs.append(None)
        gls_mse_list.append(None)

# Calculate average MSE for each model across all folds
avg_ols_mse = np.mean(ols_mse_list)
if gls_mse_list:  
    avg_gls_mse = np.mean([mse for mse in gls_mse_list if mse is not None])  
else:
    print("No MSE for FGLS.")
avg_gini_mse = np.mean(gini_mse_list)

# Calculate average estimated coefficients
avg_ols_coeff = np.mean(estimated_ols_coeffs, axis=0)
if estimated_gls_coeffs:
    avg_gls_coeff = np.mean(estimated_gls_coeffs, axis=0)
    print("GLS coef", avg_gls_coeff)
    print("OLS coef", avg_ols_coeff)
avg_gini_coeff = np.mean(estimated_gini_coeffs, axis=0)

# Create a DataFrame for comparison
if estimated_gls_coeffs:
    coeff_comparison = pd.DataFrame({
        'True Coefficients': true_coefficients,
        'Estimated OLS Coefficients': avg_ols_coeff,
        'Estimated FGLS Coefficients': avg_gls_coeff,
        'Estimated Gini Coefficients': avg_gini_coeff
    })
else:
    coeff_comparison = pd.DataFrame({
        'True Coefficients': true_coefficients,
        'Estimated OLS Coefficients': avg_ols_coeff,
        'Estimated Gini Coefficients': avg_gini_coeff
    })
print(coeff_comparison)
print("MSE OLS", avg_ols_mse)
if estimated_gls_coeffs:
    print("MSE FGLS", avg_gls_mse)
print("MSE Gini", avg_gini_mse)

In [ ]:
75000/1.1**3

In [ ]:
((56348.61*0.1*10 + 61983.47*0.1*10 + 68181.8*0.1*10 + 75000*0.1*10)+(
    56348.61*0.055*10 + 61983.47*0.055*10 + 68181.8**0.055*10 + 75000*0.055*10
    ))

In [ ]:
a = (56348.61*0.1)*(1.055)**10 
b = (a+61983.47*0.1)*(1.055)**10 
c = (b+68181.8*0.1)*1.055**10 
d = (c+75000*0.1)*(1.055)**10
d

In [ ]:
(56348.61*0.1 + 61983.47*0.1 + 68181.8*0.1 + 75000*0.1)/4

In [ ]:
# Defining constants for the new series

base = 1.055
multiplier = 5634.8
n = 40
sum_series = multiplier * ((1 - base ** n)) / (1 - base)
sum_series/15

In [ ]:
liste = []
for i in range(1,41):
    liste.append(5634*1.025**i)
np.array(liste).sum() / 15